In [1]:
import numpy as np
import pandas as pd
from efficient_probit_regression.sampling import compute_random_evaluations_probabilities_v2

# Laden der notwendigen Datensaetze
%store -r Iris
%store -r Webspam
%store -r KDDCup
%store -r Covertype
%store -r Example2D

rng = 2026
p = [2.3, 2.5, 3.0, 3.5, 4.0]
m_values = [5, 10, 50, 100, 500, 1000, 5000]
EPS = 1e-300
rows_m = []

datasets = {
    "Example2D": Example2D,
    "Iris": Iris,
    "Webspam": Webspam,
    "KDDCup": KDDCup,
    "Covertype": Covertype,
}

for dataset_name, ds in datasets.items():
    print(f"{dataset_name} Datensatz")
    print("")

    for p_val in p:
        print("p = ", p_val)
        probabilities_by_m = {}

        for m in m_values:
            probabilities_by_m[m] = compute_random_evaluations_probabilities_v2(ds.X, m=m, p=p_val, rng=rng)

        for m_from, m_to in zip(m_values[:-1], m_values[1:]):
            ratio = np.maximum(
                probabilities_by_m[m_from] / np.maximum(probabilities_by_m[m_to], EPS),
                probabilities_by_m[m_to] / np.maximum(probabilities_by_m[m_from], EPS),
            )
            height = float(np.max(ratio))

            print(f"Vergleich m={m_from} und m={m_to}")
            print(height)

            rows_m.append({
                "dataset": dataset_name,
                "p": float(p_val),
                "m_from": int(m_from),
                "m_to": int(m_to),
                "change": f"{m_from}->{m_to}",
                "height": height,
            })

        print("")


Example2D Datensatz

p =  2.3
Vergleich m=5 und m=10
437.4128604749571
Vergleich m=10 und m=50
4.7453500829518775
Vergleich m=50 und m=100
1.6120529137589539
Vergleich m=100 und m=500
1.1183040761017462
Vergleich m=500 und m=1000
1.1069689044049593
Vergleich m=1000 und m=5000
1.1006604502356048

p =  2.5
Vergleich m=5 und m=10
740.1581487466917
Vergleich m=10 und m=50
5.915096112072341
Vergleich m=50 und m=100
1.708190801063236
Vergleich m=100 und m=500
1.139888039574906
Vergleich m=500 und m=1000
1.1095204816345203
Vergleich m=1000 und m=5000
1.1070605572657493

p =  3.0
Vergleich m=5 und m=10
2812.490454795613
Vergleich m=10 und m=50
10.46020119872278
Vergleich m=50 und m=100
1.9864896380688475
Vergleich m=100 und m=500
1.2276789882708508
Vergleich m=500 und m=1000
1.1173798168237004
Vergleich m=1000 und m=5000
1.1295896963042906

p =  3.5
Vergleich m=5 und m=10
11028.497331871537
Vergleich m=10 und m=50
18.738288535933755
Vergleich m=50 und m=100
2.38253472074318
Vergleich m=100 und

In [2]:
from pathlib import Path

df = pd.DataFrame(rows_m)
df[["height"]] = df[["height"]].round(6)
df = df.sort_values(["dataset", "p", "m_from", "m_to"]).reset_index(drop=True)
display(df)

out_csv = Path("Bestimmung_m_changes_p_groesser_2.csv")
df.to_csv(out_csv, index=False)
print("Fertig! DataFrame gespeichert als:", out_csv)


,dataset,p,m_from,m_to,change,height
0,Covertype,2.3,5,10,5->10,4434.466476
1,Covertype,2.3,10,50,10->50,116.862993
2,Covertype,2.3,50,100,50->100,3.966081
3,Covertype,2.3,100,500,100->500,2.112282
4,Covertype,2.3,500,1000,500->1000,1.525823
...,...,...,...,...,...,...
145,Webspam,4.0,10,50,10->50,10953.583382
146,Webspam,4.0,50,100,50->100,150.759885
147,Webspam,4.0,100,500,100->500,36.981118
148,Webspam,4.0,500,1000,500->1000,8.547222


Fertig! DataFrame gespeichert als: Bestimmung_m_changes_p_groesser_2.csv


In [21]:
# Plots fuer die Bestimmung von m bei p > 2
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("default")
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 12

# DataFrame einlesen
df = pd.read_csv("Bestimmung_m_changes_p_groesser_2.csv")

out_dir = Path("Plots") / "Plots_Bestimmung_m_zwei_Panels_p_groesser_2"
out_dir.mkdir(exist_ok=True, parents=True)

for dataset_name in df["dataset"].unique():
    sub = df[df["dataset"] == dataset_name]

    order = (sub[["m_from", "m_to", "change"]]
             .drop_duplicates()
             .sort_values(["m_from", "m_to"]))
    x_labels = order["change"].tolist()

    fig, (ax_lin, ax_log) = plt.subplots(1, 2, figsize=(12, 5.2), sharex=True)

    for p_val in sorted(sub["p"].unique()):
        s2 = sub[sub["p"] == p_val].set_index("change").reindex(x_labels)
        y = s2["height"].values

        ax_lin.plot(x_labels, y, marker="o", label=f"p={p_val}")
        ax_log.plot(x_labels, y, marker="o", label=f"p={p_val}")

    ax_lin.set_title("Linear")
    ax_lin.grid(True, linestyle="--", alpha=0.6)

    ax_log.set_title("Log")
    ax_log.set_yscale("log")
    ax_log.grid(True, which="both", linestyle="--", alpha=0.6)

    handles, labels = ax_lin.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=min(5, len(labels)),
        bbox_to_anchor=(0.5, 0.9),
        frameon=False,
        handlelength=2.2,
    )

    for ax in (ax_lin, ax_log):
        ax.tick_params(axis="x", rotation=30)

    fig.suptitle(f"{dataset_name}", y=0.97, fontsize=20)
    fig.supxlabel("Änderung von m1 zu m2 (m1 -> m2)", fontsize=16)
    fig.supylabel("Höhe der Änderungen", fontsize=16, x=0.04)
    fig.tight_layout(rect=[0.03, 0.06, 0.98, 0.9])

    out_path = out_dir / f"{dataset_name}_two_panels_p_greater_2.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight", transparent=False)
    plt.close(fig)

    print("Gespeichert:", out_path)


Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels_p_groesser_2\Covertype_two_panels_p_greater_2.png
Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels_p_groesser_2\Example2D_two_panels_p_greater_2.png
Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels_p_groesser_2\Iris_two_panels_p_greater_2.png
Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels_p_groesser_2\KDDCup_two_panels_p_greater_2.png
Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels_p_groesser_2\Webspam_two_panels_p_greater_2.png
